# Lockbox Dataset Conversion

Dataset from Reiske et al., 2025¹, containing video and pose files of individual mice solving mechanical puzzle "lockboxes" recorded from three camera perspectives (top, front, side).

- Full dataset: https://doi.org/10.14279/depositonce-23850
- Subset of dataset (used here): https://www.dropbox.com/scl/fo/h7nkai8574h23qfq9m1b2/AP4gNZOpDJJ7z0yGtbWQiOc?rlkey=w36jzxqjkghg0j0xva5zsxy2v&st=5r9msqjw&dl=0

---

¹ Reiske, P., Boon, M. N., Andresen, N., Traverso, S., Hohlbaum, K., Lewejohann, L., Thöne-Reineke, C., Hellwich, O., & Sprekeler, H. (2025). Mouse Lockbox Dataset: Behavior Recognition for Mice Solving Lockboxes (arXiv:2505.15408). arXiv. https://doi.org/10.48550/arXiv.2505.15408


<img src="assets/lockbox1.png" width="700">

From Fig. 1 in Reiske et al., 2025¹.


In [3]:
import os 
import xarray as xr
import requests
import zipfile
from pathlib import Path
import pandas as pd
from movement.kinematics import compute_velocity, compute_speed
from movement.utils.vector import compute_norm
from movement.io import load_poses, save_poses

import ethograph as eto

### Download data

In [11]:
def download_and_extract(url: str, data_folder: Path) -> None:
    zip_path = data_folder / "dataset.zip"
    data_folder.mkdir(parents=True, exist_ok=True)
    
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(zip_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_folder)
    
    zip_path.unlink()


data_folder = eto.get_project_root() / "data" / "lockbox"
os.makedirs(data_folder, exist_ok=True)
url = "https://www.dropbox.com/scl/fo/h7nkai8574h23qfq9m1b2/AP4gNZOpDJJ7z0yGtbWQiOc?rlkey=w36jzxqjkghg0j0xva5zsxy2v&e=1&st=5r9msqjw&dl=1"

download_and_extract(url, data_folder)

data_folder = data_folder / "labeled"

### Convert dataset

In [ ]:
from pathlib import Path
import pandas as pd
import xarray as xr
import natsort

fps = 30

trials = [
    "2021-02-15_07-32-44_segment1",
    "2021-03-05_08-36-42_segment1",
    "2021-05-24_07-36-05_segment1",
    "2021-05-25_08-19-50_segment2",
    "2021-05-31_07-34-21_segment2",
    "2021-05-31_07-34-21_segment3",
]

# Explicit camera order
cameras = ["front", "side", "top"]

ds_list = []
all_video = []
all_pose = []



for trial in trials:

    files = list(data_folder.glob(f"{trial}*"))
    # containers aligned with camera order
    trial_videos = {c: None for c in cameras}
    trial_pose = {c: None for c in cameras}
    trial_datasets = {}

    for file in files:

        name = file.name

        # -----------------------------
        # VIDEO FILES
        # -----------------------------
        
        # You may want to convert to .mp4 for accurate frame seeking,
        # see https://ethograph.readthedocs.io/en/latest/troubleshooting/
        if file.suffix == ".avi":
            if "front-view" in name:
                trial_videos["front"] = name
            elif "side-view" in name:
                trial_videos["side"] = name
            elif "top-down-view" in name:
                trial_videos["top"] = name

        # -----------------------------
        # DLC POSE FILES
        # -----------------------------
        elif file.suffix == ".h5":

            df = pd.read_hdf(file)
            ds = load_poses.from_dlc_style_df(df, fps=fps)

            csv_path = str(file).replace(".h5", ".csv")
            save_poses.to_dlc_file(ds, csv_path)

            pose_name = Path(csv_path).name.replace(".csv", "_individual_0.csv")

            # -----------------------------
            # FRONT VIEW FEATURES
            # -----------------------------
            if "front-view" in name:

                ds["front_velocity"] = compute_velocity(ds.position)
                ds["front_speed"] = compute_speed(ds.position)

                trial_datasets["front"] = ds
                trial_pose["front"] = pose_name

            # -----------------------------
            # TOP VIEW FEATURES
            # -----------------------------
            elif "top-down-view" in name:

                head_centre_pos = ds.position.sel(
                    keypoints=["ear_left", "ear_right"]
                ).mean("keypoints")

                topview_pos = ds["position"]

                ds["topview_distance_head_lever_tip"] = compute_norm(
                    ds.position.sel(keypoints="lever_tip") - head_centre_pos
                )

                ds["topview_distance_head_stick_head"] = compute_norm(
                    ds.position.sel(keypoints="stick_head") - head_centre_pos
                )

                ds["topview_distance_head_ball"] = compute_norm(
                    ds.position.sel(keypoints="ball") - head_centre_pos
                )

                trial_datasets["top"] = ds
                trial_pose["top"] = pose_name

            # -----------------------------
            # SIDE VIEW (no extra features)
            # -----------------------------
            elif "side-view" in name:

                trial_datasets["side"] = ds
                trial_pose["side"] = pose_name

    # -----------------------------
    # MERGE CAMERA DATASETS
    # -----------------------------
    ds_merged = xr.merge(trial_datasets.values(), compat="override")

    # Use top view position as canonical position
    if "top" in trial_datasets:
        ds_merged["position"] = trial_datasets["top"]["position"]


    for var in ds_merged.data_vars:
        ds_merged[var].attrs["type"] = "features"

    ds_merged.attrs["trial"] = trial

  
    # append ordered camera lists
    all_video.append([trial_videos[c] for c in cameras])
    all_pose.append([trial_pose[c] for c in cameras])

    ds_list.append(ds_merged)



dt = eto.from_datasets(ds_list)

dt.set_media("video", all_video, device_labels=cameras)
dt.set_media("pose", all_pose, device_labels=cameras)


dt.save(data_folder / "lockbox.nc")